# Installation nécessaires

In [12]:
!pip install -q langgraph langchain langchain-openai requests pydantic
!pip install -q langchain-groq
!pip install -q sentence-transformers
!pip install -q chromadb rank-bm25

# Clé API

In [13]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LLM_MODEL_NAME"] = "openai/gpt-oss-120b"  # deepseek/deepseek-chat-v3-0324:free   #mistralai/mistral-small-3.2-24b-instruct:free #meta-llama/llama-3.3-70b-instruct:free #qwen/qwen3-8b:free # cohere/north-mini-code:free

print("Clé chargée :", "GROQ_API_KEY" in os.environ)
print("Modèle      :", os.environ["LLM_MODEL_NAME"])

Clé chargée : True
Modèle      : openai/gpt-oss-120b


# Vérifier les versions installés

In [14]:
import langgraph, langchain, langchain_openai
import importlib.metadata

print("langgraph       :", importlib.metadata.version("langgraph"))
print("langchain       :", langchain.__version__)
print("langchain-openai:", langchain_openai.__version__)

langgraph       : 1.2.19
langchain       : 1.4.8
langchain-openai: 1.8.0


# Importations nécessaires

In [15]:
import csv, json, operator, os, re, sys, time
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import requests
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from langchain_openai import ChatOpenAI

print("Tous les imports OK")

Tous les imports OK


# E. Phase 2 — Test du LLM + structured output (CRITIQUE)


In [16]:
from pydantic import BaseModel, Field

class Protocol(BaseModel):
    objective: str
    inclusion_criteria: list[str]
    exclusion_criteria: list[str]
    year_min: int = Field(description="année minimale de publication")

# Nouveau code (Groq)
from langchain_groq import ChatGroq
llm = ChatGroq(
    # model="openai/gpt-oss-120b",  # Un des modèles gratuits les plus performants
    model = "qwen/qwen3.8-27b",
    timeout=90,           # 90s max par requête
    max_retries=1,        # un seul retry (évite le backoff infini)
    temperature=0,
)

try:
    p = llm.with_structured_output(Protocol, method="json_schema").invoke(
        "Prépare un protocole de revue systématique sur : 'effet du café sur la cognition'."
    )
    print("✅ json_mode OK")
    print(p.model_dump_json(indent=2))
except Exception as e:
    print("❌ json_mode échoue :", type(e).__name__, e)

✅ json_mode OK
{
  "objective": "Évaluer l'effet de la consommation de café sur les fonctions cognitives chez l'adulte sain.",
  "inclusion_criteria": [
    "Essais randomisés contrôlés (ERC)",
    "Études observationnelles prospectives",
    "Participants adultes (18 ans et plus)",
    "Intervention : consommation de café (caféine ou extrait de café)",
    "Mesures de cognition : attention, mémoire, vitesse de traitement, fonctions exécutives",
    "Publication en anglais, français ou espagnol",
    "Publication entre 2010 et 2024"
  ],
  "exclusion_criteria": [
    "Études sur les animaux",
    "Études in vitro",
    "Éditoriaux",
    "Lettres à l'éditeur",
    "Études non contrôlées",
    "Études sur les populations spécifiques non représentatives (ex : patients en soins intensifs)"
  ],
  "year_min": 2010
}


# F. Phase 3 — Test OpenAlex (source de données)

In [17]:
def _abstract(inv):
    if not inv:
        return ""
    pos = {i: w for w, idxs in inv.items() for i in idxs}
    return " ".join(pos[i] for i in sorted(pos))

def search_openalex(q, n=10, year_min=2020):
    r = requests.get(
        "https://api.openalex.org/works",
        params={"search": q, "per-page": n,
                "filter": f"from_publication_date:{year_min}-01-01,has_abstract:true",
                "mailto": "ton.email@example.com"},  # pool poli
        timeout=30,
    )
    r.raise_for_status()
    return [{"title": w.get("title"), "year": w.get("publication_year"),
             "abstract": _abstract(w.get("abstract_inverted_index"))}
            for w in r.json()["results"]]

hits = search_openalex("coffee cognition", n=3)
print(f"✅ {len(hits)} résultats")
for h in hits:
    print(f"  {h['year']} — {h['title'][:80]}")

✅ 3 résultats
  2021 — Bulletproof Coffee and Cognition
  2026 — Coffee and mental health: Pharmacological effects on cognition, anxiety, depress
  2024 — Impact of coffee-derived chlorogenic acid on cognition: a systematic review and 


# G. Phase 4 — Test du graphe complet (avec réponses auto)


In [18]:
def run_auto(question, responses=None):
    """Version programmatique : réponses auto aux interrupts."""
    responses = responses or ["ok", "ok", "ok"]
    app = build_graph()
    cfg = {"configurable": {"thread_id": "test-auto"}}
    result = app.invoke({"question": question}, cfg)
    i = 0
    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value
        print(f"\n--- INTERRUPT #{i+1} ---")
        print(json.dumps(payload, ensure_ascii=False, indent=2)[:600])
        reply = responses[i] if i < len(responses) else "ok"
        print(f"→ réponse auto : {reply!r}")
        result = app.invoke(Command(resume=reply), cfg)
        i += 1
    return result

## Pipeline agentique de revue systématique — version Colab/Groq.

In [32]:
%%writefile /content/review_graph.py
"""Pipeline agentique de revue systématique — Colab/Groq + RAG Hybride (Chroma + BM25) multi-sujets."""
import csv
import hashlib
import json
import operator
import os
import re
import threading
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict

import requests
from groq import RateLimitError
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

# ============================ BLOC 1 : LLM ============================
MODEL_A = os.environ.get("MODEL_A", "openai/gpt-oss-20b")
MODEL_B = os.environ.get("MODEL_B", "qwen/qwen3.8-27b")
DEFAULT_N_PER_QUERY = int(os.environ.get("N_PER_QUERY", "2"))


def make_llm(model, **kw):
    # max_retries=0 : c'est NOTRE retry (bloc 2) qui gère les 429
    return ChatGroq(model=model, temperature=0, timeout=90, max_retries=0, **kw)


# Génération longue (planner, synthèse, critic) : reasoning bas, pas de plafond serré
llm = make_llm(MODEL_A, reasoning_effort="low")
# Screening / extraction : sorties courtes => on plafonne (max_tokens compte dans le TPM demandé)
llm_screen_a = make_llm(MODEL_A, reasoning_effort="low", max_tokens=500)
llm_screen_b = make_llm(MODEL_B, max_tokens=400)  # pas de reasoning_effort sur un modèle non-reasoning


# --- Rate limiter ---
class TokenBucket:
    def __init__(self, tpm=6000):
        self.tpm = tpm
        self.tokens = tpm
        self.last = time.time()
        self.lock = threading.Lock()

    def consume(self, n_tokens):
        with self.lock:
            now = time.time()
            self.tokens = min(self.tpm, self.tokens + (now - self.last) * (self.tpm / 60))
            self.last = now
            if self.tokens < n_tokens:
                wait = (n_tokens - self.tokens) * 60 / self.tpm
                print(f"    ⏳ TPM budget épuisé, attente {wait:.1f}s")
                time.sleep(wait)
                self.tokens = 0
            else:
                self.tokens -= n_tokens


bucket = TokenBucket(tpm=6000)  # 75 % de la limite, la fenêtre Groq est glissante

# ============================ BLOC 2 : retry + cache ============================
_WAIT_RE = re.compile(r"try again in\s+(?:(\d+)m(?!s))?\s*(?:([\d.]+)(ms|s))?", re.I)


def _retry_after(msg: str) -> float:
    m = _WAIT_RE.search(msg)
    if not m or not (m.group(1) or m.group(2)):
        return 8.0
    secs = float(m.group(2) or 0)
    if m.group(3) == "ms":
        secs /= 1000
    return int(m.group(1) or 0) * 60 + secs + 1.0


def call_llm(runnable, prompt, est_tokens, tries=6):
    """Appel LLM avec budget TPM + attente exacte demandée par Groq en cas de 429."""
    for attempt in range(1, tries + 1):
        bucket.consume(est_tokens)
        try:
            return runnable.invoke(prompt)
        except RateLimitError as e:
            wait = _retry_after(str(e))
            if wait > 120:  # quota JOURNALIER (TPD) : inutile d'attendre
                raise RuntimeError(f"Quota journalier Groq atteint : {e}") from e
            print(f"    ⏳ 429 Groq : attente {wait:.0f}s (essai {attempt}/{tries})")
            time.sleep(wait)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                wait = _retry_after(str(e))
                if wait > 120:
                    raise RuntimeError(f"Quota journalier Groq atteint : {e}") from e
                print(f"    ⏳ 429 (générique) : attente {wait:.0f}s (essai {attempt}/{tries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Rate limit persistant après plusieurs essais")


CACHE_FILE = Path("/content/llm_cache.json")
CACHE_TAG = os.environ.get("CACHE_TAG", "v1")
_cache = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else {}


def _topic_tag(question: str) -> str:
    """Isole automatiquement le cache par sujet tout en conservant le cache intra-sujet."""
    q_hash = hashlib.sha1((question or "").strip().lower().encode("utf-8")).hexdigest()[:8]
    return f"{CACHE_TAG}:{q_hash}"


def cached_call(key: str, fn, namespace: str = CACHE_TAG):
    k = f"{namespace}|{key}"
    if k not in _cache:
        _cache[k] = fn()
        CACHE_FILE.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")
    return _cache[k]


def compact_protocol(p: dict) -> str:
    """Critères seuls (sans 'objective') : ~40 % de tokens en moins par appel de screening."""
    return ("Include if: " + "; ".join(p["inclusion_criteria"]) +
            "\nExclude if: " + "; ".join(p["exclusion_criteria"]))


# ============================ RAG HYBRIDE : Chroma + BM25 ============================
_embed_model = None
_RAG_INDEXES: dict[str, dict[str, Any]] = {}


def _get_embed_model():
    """Charge sentence-transformers une seule fois en local (0 quota API)."""
    global _embed_model
    if _embed_model is None:
        from sentence_transformers import SentenceTransformer
        print("    🔧 Chargement du modèle d'embedding local (all-MiniLM-L6-v2)...")
        _embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        print("    ✅ Modèle d'embedding chargé")
    return _embed_model


def _tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", (text or "").lower())


def build_index(records):
    """Construit un index RAG hybride : base vectorielle Chroma + index lexical BM25."""
    if not records:
        return None
    import chromadb
    from rank_bm25 import BM25Okapi

    model = _get_embed_model()
    texts = [f"{r['title']}. {r['abstract']}" for r in records]
    embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

    # 1. Index dense : ChromaDB (similarité cosinus)
    client = chromadb.EphemeralClient()
    col_name = f"prisma_rag_{int(time.time() * 1000)}"
    collection = client.create_collection(
        name=col_name,
        metadata={"hnsw:space": "cosine"},
    )
    collection.add(
        ids=[str(r["id"]) for r in records],
        embeddings=embs.tolist(),
        documents=texts,
        metadatas=[
            {
                "id": int(r["id"]),
                "title": (r["title"] or "")[:200],
                "year": int(r["year"] or 0),
                "source": r.get("source", ""),
            }
            for r in records
        ],
    )

    # 2. Index lexical : BM25Okapi
    bm25 = BM25Okapi([_tokenize(t) for t in texts])

    _RAG_INDEXES[col_name] = {
        "client": client,
        "collection": collection,
        "bm25": bm25,
        "records": records,
    }
    return {
        "index_id": col_name,
        "n_docs": len(records),
        "dim": int(embs.shape[1]),
        "backend": "chroma+bm25",
    }


def retrieve_top_k(query: str, index, k: int = 2, alpha: float = 0.6):
    """Recherche hybride (alpha * Chroma cosinus + (1 - alpha) * BM25 normalisé)."""
    if not index or not query:
        return []
    store = _RAG_INDEXES.get(index["index_id"]) if isinstance(index, dict) else None
    if not store:
        return []

    records = store["records"]
    model = _get_embed_model()
    q_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]

    # Scores denses via Chroma
    n_res = min(len(records), max(k * 3, 10))
    chroma_res = store["collection"].query(
        query_embeddings=[q_emb.tolist()],
        n_results=n_res,
    )
    dense_scores = {
        int(doc_id): max(0.0, 1.0 - float(dist))
        for doc_id, dist in zip(chroma_res["ids"][0], chroma_res["distances"][0])
    }

    # Scores lexicaux via BM25
    raw_bm25 = store["bm25"].get_scores(_tokenize(query))
    max_bm25 = float(max(raw_bm25)) if len(raw_bm25) > 0 else 0.0
    bm25_scores = {
        rec["id"]: (float(score) / max_bm25 if max_bm25 > 0 else 0.0)
        for rec, score in zip(records, raw_bm25)
    }

    scored = []
    for rec in records:
        rid = rec["id"]
        d_sc = dense_scores.get(rid, 0.0)
        b_sc = bm25_scores.get(rid, 0.0)
        h_sc = alpha * d_sc + (1.0 - alpha) * b_sc
        scored.append((h_sc, d_sc, b_sc, rec))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [
        {
            "id": rec["id"],
            "title": rec["title"],
            "abstract": rec["abstract"],
            "score": float(h_sc),
            "dense_score": float(d_sc),
            "bm25_score": float(b_sc),
        }
        for h_sc, d_sc, b_sc, rec in scored[:k]
    ]


# --- Clé OpenAlex & gestion dynamique du dossier de sortie ---
OPENALEX_KEY = os.environ.get("OPENALEX_API_KEY", "")
OUT = Path("runs") / time.strftime("%Y%m%d_%H%M%S")
OUT.mkdir(parents=True, exist_ok=True)


def _get_out_dir(state: dict | None = None) -> Path:
    if state and state.get("out_dir"):
        p = Path(state["out_dir"])
        p.mkdir(parents=True, exist_ok=True)
        return p
    OUT.mkdir(parents=True, exist_ok=True)
    return OUT


class ReviewState(TypedDict, total=False):
    question: str
    n_per_query: int
    out_dir: str
    human_feedback: str
    protocol: dict
    queries: list[str]
    records: list[dict]
    index: Any
    screening: list[dict]
    included: list[dict]
    extraction: list[dict]
    report: str
    critic_issues: list[str]
    revisions: int
    prisma: dict
    log: Annotated[list[dict], operator.add]


def trace(agent: str, event: str, data=None, state: dict | None = None) -> dict:
    entry = {"t": time.time(), "agent": agent, "event": event, "data": data}
    out_dir = _get_out_dir(state)
    with open(out_dir / "trace.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False, default=str) + "\n")
    return entry


class Protocol(BaseModel):
    objective: str
    inclusion_criteria: list[str]
    exclusion_criteria: list[str]
    year_min: int = Field(description="minimum publication year")


class Queries(BaseModel):
    queries: list[str] = Field(description="4 to 5 short English keyword queries covering all facets of the question")


class ScreenDecision(BaseModel):
    decision: Literal["include", "exclude"]
    criterion: str = Field(description="criterion used, max 10 words")
    rationale: str = Field(description="max 20 words")


class Extraction(BaseModel):
    study_design: str
    data_or_population: str
    method: str
    key_results: str
    limitations: str
    quality_score: int = Field(ge=0, le=5)
    quality_rationale: str


class CriticVerdict(BaseModel):
    ok: bool
    issues: list[str]


def _abstract(inv):
    if not inv:
        return ""
    pos = {i: w for w, idxs in inv.items() for i in idxs}
    return " ".join(pos[i] for i in sorted(pos))


def search_openalex(q, n=DEFAULT_N_PER_QUERY, year_min=2018, max_retries=3):
    url = "https://api.openalex.org/works"
    headers = {"Authorization": f"Bearer {OPENALEX_KEY}"} if OPENALEX_KEY else {}
    params = {
        "search": q,
        "per-page": n,
        "filter": f"from_publication_date:{year_min}-01-01",
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            results = r.json().get("results", [])
            print(f"    OpenAlex '{q[:50]}' → {len(results)} hits")
            return [{"source": "openalex", "title": w.get("title") or "", "doi": w.get("doi"),
                     "year": w.get("publication_year"), "url": w.get("id"),
                     "abstract": _abstract(w.get("abstract_inverted_index"))}
                    for w in results]
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
    print(f"    ❌ OpenAlex abandonné : {q[:60]}")
    return []


def _simplify_query(q: str) -> str:
    """Réduit une requête booléenne complexe à des mots-clés simples tout en préservant les acronymes techniques (ex: GLiNER, LoRA, NER)."""
    q = re.sub(r'[()"\']', ' ', q)
    q = re.sub(r'\b(AND|OR|NOT)\b', ' ', q, flags=re.IGNORECASE)
    q = re.sub(r'publication_year\s*:\s*[<>=]+\s*\d+', ' ', q)
    q = re.sub(r'\byear\s*[<>=]+\s*\d+', ' ', q)
    words = [w for w in q.split() if len(w) >= 3][:8]
    return " ".join(words)


def search_arxiv(q, n=DEFAULT_N_PER_QUERY, year_min=2018, max_retries=3):
    """Recherche sur l'API Atom officielle d'arXiv."""
    url = "https://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{q}",
        "start": 0,
        "max_results": n,
        "sortBy": "relevance",
        "sortOrder": "descending",
    }
    headers = {"User-Agent": "SystematicReviewAgent/1.0 (Colab)"}
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
    }
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            root = ET.fromstring(r.text)
            hits = []
            for entry in root.findall("atom:entry", ns):
                published = entry.findtext("atom:published", "", ns)
                year = int(published[:4]) if published[:4].isdigit() else None
                if year and year < year_min:
                    continue
                title = " ".join(entry.findtext("atom:title", "", ns).split())
                abstract = " ".join(entry.findtext("atom:summary", "", ns).split())
                entry_url = entry.findtext("atom:id", "", ns).strip()
                doi_raw = entry.findtext("arxiv:doi", "", ns).strip()
                doi = f"https://doi.org/{doi_raw}" if doi_raw else None
                hits.append({
                    "source": "arxiv",
                    "title": title,
                    "doi": doi,
                    "year": year,
                    "url": entry_url,
                    "abstract": abstract,
                })
            print(f"    arXiv    '{q[:50]}' → {len(hits)} hits")
            return hits
        except (requests.exceptions.RequestException, ET.ParseError):
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
    print(f"    ❌ arXiv abandonné : {q[:60]}")
    return []


def planner(state):
    # Initialise un dossier dédié au sujet courant si non défini
    out_dir = state.get("out_dir")
    if not out_dir:
        slug = re.sub(r"[^a-z0-9]+", "_", state["question"].lower()).strip("_")[:35]
        p = Path("runs") / f"{time.strftime('%Y%m%d_%H%M%S')}_{slug}"
        p.mkdir(parents=True, exist_ok=True)
        out_dir = str(p)

    state_with_dir = {**state, "out_dir": out_dir}
    feedback = state.get("human_feedback", "")
    prompt = (
        f"You are preparing a systematic review (PRISMA 2020).\n"
        f"Research question: {state['question']}\n"
        f"User feedback to integrate: {feedback or 'none'}\n"
        "Produce the protocol: objective, inclusion criteria, exclusion criteria, minimum year.\n"
        "IMPORTANT:\n"
        "- Write everything in ENGLISH.\n"
        "- Adapt the criteria strictly to the research domain (e.g., empirical benchmarks/datasets/models "
        "for computer science/NLP, or clinical/observational cohorts for biomedical topics).\n"
        "- Specify ONLY OpenAlex and arXiv as searched databases (do NOT mention PubMed or other unqueried sources)."
    )
    structured_planner = llm.with_structured_output(Protocol, method="json_schema")
    protocol = call_llm(structured_planner, prompt, est_tokens=1500).model_dump()
    return {
        "out_dir": out_dir,
        "protocol": protocol,
        "log": [trace("planner", "protocol", protocol, state_with_dir)],
    }


def human_protocol(state):
    answer = interrupt({"validate_protocol": state["protocol"],
                        "reply": "ok, or your comments to revise"})
    if str(answer).strip().lower() == "ok":
        return Command(goto="search_strategy",
                       update={"log": [trace("human", "protocol_approved", state=state)]})
    return Command(goto="planner", update={"human_feedback": str(answer),
                                           "log": [trace("human", "protocol_feedback", answer, state)]})


def search_strategy(state):
    prompt = (
        f"Research question: {state['question']}\n"
        f"Protocol: {json.dumps(state['protocol'], ensure_ascii=False)}\n"
        "Generate 4 to 5 targeted search queries for OpenAlex and arXiv.\n"
        "IMPORTANT:\n"
        "- Write queries in ENGLISH only.\n"
        "- Use SHORT keyword phrases (3-6 words), NO Boolean operators (AND/OR/NOT) and NO parentheses.\n"
        "- Do NOT include year filters or language filters (handled separately).\n"
        "- Cover all distinct methods, models, and evaluation dimensions mentioned in the research question "
        "(e.g., one query per model family or evaluation aspect)."
    )
    structured_search = llm.with_structured_output(Queries, method="json_schema")
    queries = call_llm(structured_search, prompt, est_tokens=1500).queries
    return {"queries": queries, "log": [trace("search_strategy", "queries", queries, state)]}


def retrieval(state):
    raw = []
    year_min = state["protocol"]["year_min"]
    n_hits = int(state.get("n_per_query") or DEFAULT_N_PER_QUERY)
    for q in state["queries"]:
        simple = _simplify_query(q) or q
        found_oa = search_openalex(simple, n=n_hits, year_min=year_min)
        found_ax = search_arxiv(simple, n=n_hits, year_min=year_min)
        found = found_oa + found_ax
        trace("retrieval", "query_hits", {
            "query": q, "simplified": simple, "openalex": len(found_oa), "arxiv": len(found_ax)
        }, state)
        raw += found
        time.sleep(0.3)
    seen, records = set(), []
    for r in raw:
        if not r["abstract"] or len(r["abstract"]) < 100:
            continue
        key = (r["doi"] or re.sub(r"\W+", "", (r["title"] or "").lower()))
        if key and key not in seen:
            seen.add(key)
            records.append({**r, "id": len(records) + 1})

    print("    🔍 Construction de l'index RAG hybride (Chroma + BM25)...")
    index = build_index(records)
    if index:
        print(f"    ✅ Index RAG prêt : {index['n_docs']} documents (Chroma dim={index['dim']} + BM25)")

    prisma = {"identified": len(raw), "duplicates_removed": len(raw) - len(records)}
    return {
        "records": records,
        "index": index,
        "prisma": prisma,
        "log": [
            trace("retrieval", "done", prisma, state),
            trace("retrieval", "index_built", index, state),
        ],
    }


REVIEWER_STYLES = {
    "A": "You are a STRICT reviewer: exclude if any inclusion criterion is uncertain.",
    "B": "You are an INCLUSIVE reviewer: include if in doubt — full text will decide.",
}


# ============================ BLOC 3 : screening + extraction ============================
def screening(state):
    runnables = {
        "A": llm_screen_a.with_structured_output(ScreenDecision, method="json_schema"),
        "B": llm_screen_b.with_structured_output(ScreenDecision, method="function_calling"),
    }
    ns = _topic_tag(state["question"])
    proto = compact_protocol(state["protocol"])
    results, included = [], []
    for i, rec in enumerate(state["records"], 1):
        votes = {}
        for name, style in REVIEWER_STYLES.items():
            prompt = (f"{style}\n{proto}\nTitle: {rec['title']}\n"
                      f"Abstract: {rec['abstract'][:600]}")
            votes[name] = cached_call(
                f"screen|{name}|{rec['title']}",
                lambda n=name, p=prompt: call_llm(
                    runnables[n], p, len(p) // 3 + 250
                ).model_dump(),
                namespace=ns,
            )
        a, b = votes["A"]["decision"], votes["B"]["decision"]
        final = "include" if "include" in (a, b) else "exclude"
        results.append({"id": rec["id"], "title": rec["title"], "A": votes["A"],
                        "B": votes["B"], "conflict": a != b, "final": final})
        if final == "include":
            included.append(rec)
        print(f"    [{i}/{len(state['records'])}] {final:7s} {rec['title'][:60]}")
    prisma = {**state["prisma"], "screened": len(state["records"]),
              "excluded": len(state["records"]) - len(included), "included": len(included)}
    out_dir = _get_out_dir(state)
    with open(out_dir / "screening.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return {"screening": results, "included": included, "prisma": prisma,
            "log": [trace("screening", "done", prisma, state)]}


def human_screening(state):
    conflicts = [s["title"] for s in state["screening"] if s["conflict"]][:5]
    answer = interrupt({"prisma": state["prisma"], "conflict_examples": conflicts,
                        "reply": "ok to continue"})
    return {"log": [trace("human", "screening_reviewed", answer, state)]}


def extraction(state):
    if not state.get("included"):
        return {"extraction": [], "log": [trace("extraction", "skipped_no_included", state=state)]}
    ns = _topic_tag(state["question"])
    structured = llm_screen_a.with_structured_output(Extraction, method="json_schema")
    table = []
    for rec in state["included"]:
        rag_hits = retrieve_top_k(f"{state['question']} {rec['title']}", state.get("index"), k=2)
        rag_context = ""
        if rag_hits:
            rag_context = "\nRelated context (Chroma+BM25 cross-references): " + " | ".join(
                f"[{h['id']}] {h['title'][:75]} (hybrid={h['score']:.2f}, dense={h['dense_score']:.2f}, bm25={h['bm25_score']:.2f})"
                for h in rag_hits
            )
            print(
                f"    🔍 RAG (Chroma+BM25) pour '{rec['title'][:38]}' : "
                f"{[(h['id'], round(h['score'], 2)) for h in rag_hits]}"
            )

        prompt = (
            "Extract structured data relevant to the research question from this study. "
            "Answer in ENGLISH, keep each field under 25 words.\n"
            f"Research question: {state['question']}\n"
            f"Title: {rec['title']}\nAbstract: {rec['abstract'][:600]}{rag_context}"
        )
        row = cached_call(
            f"extract-rag|{rec['title']}",
            lambda p=prompt: call_llm(structured, p, len(p) // 3 + 400).model_dump(),
            namespace=ns,
        )
        table.append({"id": rec["id"], "title": rec["title"], "year": rec["year"], **row})
    return {"extraction": table, "log": [trace("extraction", "done", {"n": len(table)}, state)]}


def _build_references_section(included: list[dict]) -> str:
    """Génère une bibliographie vérifiée à partir des études incluses."""
    lines = ["\n## References\n"]
    for r in included:
        link = r.get("doi") or r.get("url") or ""
        src = r.get("source", "").upper()
        year = r.get("year") or "n.d."
        ref = f"- **[{r['id']}]** {r['title']} ({year}). *{src}*."
        if link:
            ref += f" {link}"
        lines.append(ref)
    return "\n".join(lines) + "\n"


def _deliver_outputs(state: dict, report: str):
    """Écrit et livre sur disque le rapport Markdown, le tableau CSV et les stats PRISMA."""
    out_dir = _get_out_dir(state)
    report_path = out_dir / "report.md"
    colab_copy = Path("/content/systematic_review_report.md")
    report_path.write_text(report, encoding="utf-8")
    colab_copy.write_text(report, encoding="utf-8")

    if state.get("prisma"):
        (out_dir / "prisma.json").write_text(
            json.dumps(state["prisma"], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    extraction_rows = state.get("extraction") or []
    if extraction_rows:
        csv_path = out_dir / "extraction.csv"
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(extraction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(extraction_rows)
    print(f"    📄 Synthèse livrée : {report_path} (et {colab_copy})")


def synthesis(state):
    if not state.get("extraction"):
        empty_report = "# Systematic Literature Review\n\nNo included studies — cannot synthesize."
        _deliver_outputs(state, empty_report)
        return {"report": empty_report,
                "revisions": state.get("revisions", 0) + 1,
                "log": [trace("synthesis", "skipped_no_extraction", state=state)]}

    issues = state.get("critic_issues") or []
    prisma = state.get("prisma") or {}
    n_included = len(state["extraction"])
    global_rag = retrieve_top_k(state["question"], state.get("index"), k=min(5, n_included))
    rag_summary = ", ".join(f"[{h['id']}] (score={h['score']:.2f})" for h in global_rag)

    prompt = (
        "Write a complete, publication-grade systematic literature review in ENGLISH (Markdown).\n"
        f"Research question: {state['question']}\n"
        f"Exact number of included studies: {n_included}\n"
        f"PRISMA 2020 counts: {json.dumps(prisma, ensure_ascii=False)}\n"
        f"Top RAG-ranked studies (Chroma + BM25): {rag_summary}\n"
        f"Evidence table (quality_score is on a 0 to 5 scale): {json.dumps(state['extraction'], ensure_ascii=False)}\n\n"
        "Required structure:\n"
        "# Systematic Literature Review: <concise title>\n"
        "## 1. Executive Summary\n"
        "## 2. Methodology & PRISMA Flow (databases searched: OpenAlex and arXiv only; report exact PRISMA counts)\n"
        "## 3. Characteristics & Quality of Included Studies (include a Markdown comparison table citing [id] and quality_score/5)\n"
        "## 4. Thematic & Comparative Synthesis (organize around the key methods/dimensions of the research question, cite every claim with [id])\n"
        "## 5. Limitations & Future Directions\n\n"
        "Rules:\n"
        "- State the exact count of included studies consistently across all sections.\n"
        "- Cite ONLY valid study IDs using [id] matching the evidence table.\n"
        "- Do NOT invent a References section (it will be appended automatically).\n"
        f"- Issues to fix from previous critic review: {issues or 'none'}"
    )
    draft = call_llm(llm, prompt, est_tokens=len(prompt) // 3 + 1800).content
    draft_clean = re.split(r"\n##\s*References\b", draft, flags=re.IGNORECASE)[0].rstrip()
    full_report = draft_clean + "\n" + _build_references_section(state.get("included") or [])

    _deliver_outputs(state, full_report)
    rev = state.get("revisions", 0) + 1
    return {
        "report": full_report,
        "revisions": rev,
        "log": [trace("synthesis", "draft_delivered", {"revision": rev, "chars": len(full_report)}, state)],
    }


def critic(state):
    if not state.get("extraction"):
        return {"critic_issues": [], "log": [trace("critic", "skipped_empty", state=state)]}
    valid = {str(r["id"]) for r in state["included"]}
    body = re.split(r"\n##\s*References\b", state["report"], flags=re.IGNORECASE)[0]
    cited = set(re.findall(r"\[(\d+)\]", body))
    issues = [f"Unknown citation [{c}]" for c in cited - valid]
    critic_prompt = (
        f"Protocol: {json.dumps(state['protocol'], ensure_ascii=False)}\n"
        f"Included studies count: {len(state['included'])}\n"
        f"Report: {body}\n"
        "Check ONLY fixable factual errors in the report: "
        "1) consistency with the protocol and exact PRISMA counts, "
        "2) unsupported claims or overstatements, "
        "3) accurate quality_score reporting (scale is 0 to 5). "
        "Note: inline numeric citations [id] matching the evidence table are the required format (do NOT flag [id] formatting), "
        "and do NOT flag inherent small sample size of the retrieved corpus."
    )
    structured_critic = llm.with_structured_output(CriticVerdict, method="json_schema")
    verdict = call_llm(structured_critic, critic_prompt, est_tokens=1500)
    issues += verdict.issues
    return {"critic_issues": issues, "log": [trace("critic", "verdict", issues, state)]}


def route_after_critic(state):
    if not state.get("extraction"):
        return "end"
    return "end" if not state["critic_issues"] or state["revisions"] >= 2 else "synthesis"


def build_graph():
    g = StateGraph(ReviewState)
    for name, fn in [("planner", planner), ("human_protocol", human_protocol),
                     ("search_strategy", search_strategy), ("retrieval", retrieval),
                     ("screening", screening), ("human_screening", human_screening),
                     ("extraction", extraction), ("synthesis", synthesis), ("critic", critic)]:
        g.add_node(name, fn)
    g.add_edge(START, "planner")
    g.add_edge("planner", "human_protocol")
    g.add_edge("search_strategy", "retrieval")
    g.add_edge("retrieval", "screening")
    g.add_edge("screening", "human_screening")
    g.add_edge("human_screening", "extraction")
    g.add_edge("extraction", "synthesis")
    g.add_edge("synthesis", "critic")
    g.add_conditional_edges("critic", route_after_critic, {"synthesis": "synthesis", "end": END})
    return g.compile(checkpointer=MemorySaver())

Overwriting /content/review_graph.py


## 1. Relance le module :



In [33]:
import importlib, sys
sys.path.insert(0, "/content")
import review_graph
importlib.reload(review_graph)
print("✅ Module rechargé")

✅ Module rechargé


# Lancer le script de bout en bout

In [ ]:
import importlib
import json
import sys
import time
from IPython.display import Markdown, display

sys.path.insert(0, "/content")
import review_graph
importlib.reload(review_graph)


def run_auto(question, responses=None, n_per_query=2):
    responses = responses or ["ok", "ok", "ok"]
    app = review_graph.build_graph()
    cfg = {"configurable": {"thread_id": f"review-{int(time.time())}"}}
    result = app.invoke({"question": question, "n_per_query": n_per_query}, cfg)
    i = 0
    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value
        print(f"\n--- INTERRUPT #{i+1} ---")
        print(json.dumps(payload, ensure_ascii=False, indent=2)[:800])
        reply = responses[i] if i < len(responses) else "ok"
        print(f"→ réponse auto : {reply!r}")
        result = app.invoke(review_graph.Command(resume=reply), cfg)
        i += 1
    return result


question_ner = (
    "Comparison of information extraction approaches (fine-tuned encoders, GLiNER, "
    "prompted LLMs, LoRA fine-tuned LLMs) for named entity recognition: "
    "accuracy, cost, and out-of-domain generalization"
)

t0 = time.time()
result = run_auto(question_ner, responses=["ok", "ok"], n_per_query=2)
print(f"\n⏱️ Temps total : {time.time() - t0:.1f}s")
print("Dossier     :", result.get("out_dir"))
print("PRISMA      :", result.get("prisma"))
print("Inclus      :", len(result.get("included", [])))
print("Extraits    :", len(result.get("extraction", [])))
print("Révisions   :", result.get("revisions"))
print("Issues      :", result.get("critic_issues"))

display(Markdown(result.get("report") or ""))


--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To compare the performance of different information extraction approaches—fine‑tuned encoders, GLiNER, prompted large language models (LLMs), and LoRA‑fine‑tuned LLMs—for named entity recognition (NER) in terms of accuracy, computational cost, and out‑of‑domain generalization.",
    "inclusion_criteria": [
      "Studies reporting empirical benchmarks on NER tasks using at least one of the following approaches: fine‑tuned encoders, GLiNER, prompted LLMs, or LoRA‑fine‑tuned LLMs.",
      "Publications that provide quantitative metrics for accuracy (e.g., F1, precision, recall) and/or computational cost (e.g., inference time, GPU hours, memory usage).",
      "Studies that evaluate out‑of‑domain generalization, either through cross‑dataset evaluat
→ réponse auto : 'ok'
    OpenAlex 'fine tuned encoder named entity recognition' → 2 hits
    arXiv    'fine tuned encoder named entity recognition' → 2 hits
    OpenAlex 'GLiNE

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    ✅ Modèle d'embedding chargé
    ✅ Index RAG prêt : 17 documents (Chroma dim=384 + BM25)
    [1/17] include A Fine-Tuned Bidirectional Encoder Representations From Tran
    [2/17] include BanglaCoNER: Towards Robust Bangla Complex Named Entity Reco
    [3/17] include Query-Based Named Entity Recognition
    [4/17] include Named Entity Recognition with GLiNER and Relation Extraction
    [5/17] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    [6/17] include GLiNER-Relex: A Unified Framework for Joint Named Entity Rec
    [7/17] include RLSP-NER: Reinforcement Learning of Soft Prompts for NER wit
    [8/17] include ZeroTuneBio NER: A three-stage framework for zero-shot and z
    [9/17] exclude Learning From Failure: Integrating Negative Examples when Fi
    [10/17] exclude Demystifying Instruction Mixing for Fine-tuning Large Langua
    [11/17] include LLM-NER: Advancing Named Entity Recognition with LoRA+ Fine-
    [12/17] exclude Federated Sketching LoRA: A Fl

In [12]:
t0 = time.time()
result_cached = run_auto(question_ner, responses=["ok", "ok"], n_per_query=2)
print(f"⏱️ Temps avec cache : {time.time() - t0:.1f}s")


--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To systematically compare information extraction approaches—fine‑tuned encoder models, GLiNER, prompted large language models (LLMs), and LoRA‑fine‑tuned LLMs—for named entity recognition (NER) in terms of accuracy, computational cost, and out‑of‑domain generalization.",
    "inclusion_criteria": [
      "Primary research articles that evaluate at least one of the specified NER approaches on a benchmark dataset.",
      "Studies reporting quantitative performance metrics for NER (e.g., precision, recall, F1‑score) and/or computational cost (e.g., inference time, GPU hours, monetary cost).",
      "Experiments that assess out‑of‑domain or cross‑domain generalization (e.g., training on one corpus and testing on a distinct corpus).",
      "Publica
→ réponse auto : 'ok'
    OpenAlex 'fine tuned encoder NER' → 2 hits
    arXiv    'fine tuned encoder NER' → 2 hits
    OpenAlex 'GLiNER named entity extraction' → 2 hits
    ar

# Tester la boucle interactive (Human-in-the-Loop)

In [13]:
result_hitl = run_auto(
    question_ner,
    responses=[
        "Restrict minimum publication year to 2022 and require zero-shot or few-shot evaluation.",
        "ok",  # approuve le protocole révisé
        "ok",  # approuve le screening
    ],
    n_per_query=2,
)
print("Année min après révision :", result_hitl["protocol"]["year_min"])


--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To systematically compare information extraction approaches—fine‑tuned encoder models, GLiNER, prompted large language models (LLMs), and LoRA‑fine‑tuned LLMs—for named entity recognition (NER) in terms of accuracy, computational cost, and out‑of‑domain generalization.",
    "inclusion_criteria": [
      "Primary research articles that evaluate at least one of the specified NER approaches on a benchmark dataset.",
      "Studies reporting quantitative performance metrics for NER (e.g., precision, recall, F1‑score) and/or computational cost (e.g., inference time, GPU hours, monetary cost).",
      "Experiments that assess out‑of‑domain or cross‑domain generalization (e.g., training on one corpus and testing on a distinct corpus).",
      "Publica
→ réponse auto : 'Restrict minimum publication year to 2022 and require zero-shot or few-shot evaluation.'

--- INTERRUPT #2 ---
{
  "validate_protocol": {
    "objective": "To 

In [14]:
import pandas as pd
from pathlib import Path

out_dir = Path(result["out_dir"])
print("Fichiers générés :", [p.name for p in out_dir.iterdir()])

# 1. Tableau d'extraction structuré
df_ext = pd.read_csv(out_dir / "extraction.csv")
display(df_ext[["id", "year", "title", "method", "quality_score"]])

# 2. Taux d'accord inter-relecteurs (A vs B) sur le screening
screen_data = json.loads((out_dir / "screening.json").read_text(encoding="utf-8"))
agreements = sum(1 for s in screen_data if not s["conflict"])
print(f"Accord brut Relecteur A / B : {agreements}/{len(screen_data)} ({agreements/len(screen_data):.1%})")

Fichiers générés : ['screening.json', 'prisma.json', 'extraction.csv', 'report.md', 'trace.jsonl']


,id,year,title,method,quality_score
0,1,2026,Uzbek Named Entity Recognition: In-Domain Fine...,"In-domain fine-tuned XLM‑R‑large, CRF baseline...",4
1,2,2026,Domain-Adaptive Stressor Identification Using ...,Fine‑tuned BERT model trained on the corpus to...,3
2,5,2025,Named Entity Recognition with GLiNER and Relat...,Fine‑tuned GLiNER encoder on the dataset; comp...,3
3,6,2026,GLiNER-Relex: A Unified Framework for Joint Na...,Extended GLiNER with shared transformer encode...,3
4,7,2026,GLiNER-Relex: A Unified Framework for Joint Na...,Extended GLiNER with shared transformer encode...,3
5,8,2019,Query-Based Named Entity Recognition,Cast NER as query‑based MRC; queries encode en...,3
6,9,2026,RLSP-NER: Reinforcement Learning of Soft Promp...,Two‑stage: LoRA‑fine‑tuned model with manual p...,3
7,10,2025,ZeroTuneBio NER: A three-stage framework for z...,Three‑stage framework: chain‑of‑thought prompt...,4
8,13,2025,LLM-NER: Advancing Named Entity Recognition wi...,LoRA and LoRA+ adaptation applied to Meta‑Llam...,3
9,16,2022,Searching for Optimal Subword Tokenization in ...,Introduced X-Piece subword tokenization; compa...,3


Accord brut Relecteur A / B : 19/19 (100.0%)


## Étape 4 — Lancer le graphe complet


In [ ]:
# complet
import importlib, sys, json
sys.path.insert(0, "/content")
import review_graph
importlib.reload(review_graph)


def run_auto(question, responses=None):
    responses = responses or ["ok", "ok", "ok"]
    app = review_graph.build_graph()
    cfg = {"configurable": {"thread_id": f"test-auto-{int(time.time())}"}}
    result = app.invoke({"question": question}, cfg)
    i = 0
    while result.get("__interrupt__"):
        payload = result["__interrupt__"][0].value
        print(f"\n--- INTERRUPT #{i+1} ---")
        print(json.dumps(payload, ensure_ascii=False, indent=2)[:800])
        reply = responses[i] if i < len(responses) else "ok"
        print(f"→ réponse auto : {reply!r}")
        result = app.invoke(review_graph.Command(resume=reply), cfg)
        i += 1
    return result


import time
result = run_auto("Effects of coffee on cognitive performance", responses=["ok", "ok"])

print("\n" + "="*60)
print("PRISMA      :", result.get("prisma"))
print("Inclus      :", len(result.get("included", [])))
print("Extraits    :", len(result.get("extraction", [])))
print("Révisions   :", result.get("revisions"))
print("Issues      :", result.get("critic_issues"))
print("\nRapport (extrait) :")
print((result.get("report") or "")[:800])


--- INTERRUPT #1 ---
{
  "validate_protocol": {
    "objective": "To systematically review and synthesize the evidence on the effects of coffee consumption on cognitive performance in humans.",
    "inclusion_criteria": [
      "Original peer‑reviewed research articles (randomized controlled trials, non‑randomized controlled trials, cohort studies, case‑control studies, cross‑sectional studies) that assess coffee intake as an exposure.",
      "Studies that measure cognitive performance using validated neuropsychological tests or objective cognitive outcomes (e.g., memory, attention, executive function, processing speed).",
      "Human participants of any age, sex, or health status.",
      "Studies published in English.",
      "Full‑text articles available through OpenAlex, arXiv, PubMed, or other scholarly
→ réponse auto : 'ok'
    OpenAlex 'coffee cognitive performance' → 2 hits
    arXiv    'coffee cognitive performance' → 2 hits
    OpenAlex 'coffee memory attention' → 2 hits
 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    ✅ Modèle d'embedding chargé
    ✅ Index RAG prêt : 19 documents (Chroma dim=384 + BM25)
    [1/19] include Coffee, Caffeine, and Cognitive Performance
    [2/19] include Association of Coffee, Decaffeinated Coffee and Caffeine Int
    ⏳ TPM budget épuisé, attente 1.0s
    [3/19] exclude Cognition and Emotion: Perspectives of a Closing Gap
    ⏳ TPM budget épuisé, attente 6.8s
    ⏳ TPM budget épuisé, attente 1.3s
    [4/19] exclude Cognitive computation with autonomously active neural networ
    ⏳ TPM budget épuisé, attente 6.6s
    ⏳ TPM budget épuisé, attente 1.3s
    [5/19] exclude Coffee, attention, memory and mood: from the brain to the wo
    ⏳ TPM budget épuisé, attente 6.0s
    ⏳ TPM budget épuisé, attente 2.1s
    [6/19] exclude Learning Efficient Algorithms with Hierarchical Attentive Me
    ⏳ TPM budget épuisé, attente 5.9s
    ⏳ TPM budget épuisé, attente 1.9s
    [7/19] exclude Runtime Observability for Heterogeneous Attention Memory
    ⏳ TPM budget épuisé, attente 6.